# CamShift 객체 추적

## 개요
- **CamShift** : MeanShift의 업그레이드 버전
- 객체의 **크기가 바뀌거나 회전**해도 추적 가능
- MeanShift와의 차이점
  - MeanShift : **고정 크기** 박스로 추적
  - CamShift : **적응형 크기 + 회전 가능한** 박스로 추적

## 동작 흐름
```
첫 프레임 → ROI 지정 → HSV 변환 → 히스토그램 계산
                                         ↓
매 프레임 → HSV 변환 → 역투영(Back Projection) → CamShift → 회전 박스 시각화
```

In [18]:
import numpy as np
import cv2
from google.colab.patches import cv2_imshow
import time

# 추적 상태 변수
roi_hist = None
track_window = None

# 초기 추적 영역 (ROI) 좌표 설정 함수가 잇긴 한데 코랩에서 안됨 그래서 하드코딩.
x,y,w,h = 70,280,100,100

# 종료조건
# 아래 두 조건 중 하나라도 충족되면 반복 종료
# cv2.TERM_CRITERIA_EPS  : 이동 거리가 1픽셀 이하일 때
# cv2.TERM_CRITERIA_COUNT: 반복 횟수가 10회를 초과할 때

termination = (cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 10, 1) # 순서가 반대네요?
print(f'초기 추적 대상 영역 설정 완료 : (x = {x}, y= {y}, w = {w}, h = {h})')


초기 추적 대상 영역 설정 완료 : (x = 70, y= 280, w = 100, h = 100)


In [5]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
video_path = '/content/drive/MyDrive/CV_work/AI_응용_강의자료/data/top-down.mp4'

cap = cv2.VideoCapture(video_path)



if not cap.isOpened():
    cap.release()
    raise FileNotFoundError(
        f"\n비디오 파일을 열 수 없습니다.\n"
        f"   경로를 확인하세요: {video_path}\n"
        f"   파일이 없다면 Google Drive에 업로드하거나 경로를 수정하세요."

    )

In [20]:
# 영상 기본 정보 출력 : 여기저기 복붙해서 쓰기 좋음

total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps          = cap.get(cv2.CAP_PROP_FPS)
width_vid    = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height_vid   = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'   해상도: {width_vid} x {height_vid}')
print(f'   FPS   : {fps:.1f}')
print(f'   총 프레임: {total_frames}')

   해상도: 1920 x 1080
   FPS   : 30.0
   총 프레임: 405


In [21]:
# 출력 제한 설정.
frame_count = 0
MAX_FRAME_TO_PROCESS= 150 # 최대 150 프레임 -> 이미지 150장 까지 처리
DISPLAY_EVERY_N_FRAMES = 20 # 20 프레임 마다 결과 출력


print(f'MeanShift 추적 시작(최대 {MAX_FRAME_TO_PROCESS} 프레임, {DISPLAY_EVERY_N_FRAMES} 프레임 마다 출력 )')



MeanShift 추적 시작(최대 150 프레임, 20 프레임 마다 출력 )


## ROI 히스토그램 등록

### 노이즈 필터링 마스크
- HSV 색공간에서 추적에 방해되는 픽셀을 제거
  - **하한** `(0, 50, 50)` : H=모든 색상 허용, S≥50, V≥50
  - **상한** `(180, 255, 255)`

| 채널 | 의미 | 제거 대상 |
|------|------|-----------|
| H (Hue, 색상) | 0~180° | 모든 색상 허용 |
| S (Saturation, 채도) | 0~255 | S < 50 : 회색 계열 → 추적 혼란 유발 |
| V (Value, 명도) | 0~255 | V < 50 : 너무 어두움 → 검정에 가까움 |

In [22]:
# 첫 프레임 읽기
ret, frame = cap.read()
if not ret:
    cap.release()
    raise RuntimeError(
        '첫 프레임을 읽을 수 없습니다.\n'
        '   비디오 경로가 올바른지, 파일이 손상되지 않았는지 확인하세요.'
    )


# --- ROI 크기 유효성 검사 --- # 해도되고 안해도 된대요.
frame_h, frame_w = frame.shape[:2]
if w <= 0 or h <= 0:
    raise ValueError(f'ROI 크기 오류: w={w}, h={h} (0보다 커야 합니다)')
if x < 0 or y < 0 or x + w > frame_w or y + h > frame_h:
    raise ValueError(
        f'ROI 범위 초과: 프레임({frame_w}x{frame_h})를 벗어납니다.\n'
        f'   설정값: x={x}, y={y}, w={w}, h={h}\n'
        f'   x+w={x+w}, y+h={y+h}'
    )

In [23]:
 # ROI추출 및 HSV 변환

roi = frame[y:y+h,x:x+w]
roi_hsv = cv2.cvtColor(roi,cv2.COLOR_RGB2HSV)

# 마스크 생성 : 채도, 명도 낮은 픽셀 제거

mask = cv2.inRange(roi_hsv, np.array([0.,50.,50]), np.array([180.,255.,255.]))

# 이게 거의 디폴트값이라 이것도 복붙한 다음에 조정 해주면 됨.

# H채널 히스토그램 계산 -> 정규화 (0-255)

roi_hist = cv2.calcHist([roi_hsv],[0],mask,[180],[0,180])

cv2.normalize(roi_hist, roi_hist, 0,255,cv2.NORM_MINMAX)

# 초기 추적 윈도우 설정
track_window = (x,y,w,h)

# 첫 프레임 ROI 박스 표시 후 확인
first_frame_display = frame.copy()
cv2.rectangle(first_frame_display, (x,y),(x+w,y+h),(0,255,0),2)
cv2.putText(first_frame_display, 'Initial ROI', (x,y-10),cv2.FONT_HERSHEY_SIMPLEX,0.7,
            (0,0,255),2,cv2.LINE_AA)




cv2_imshow(first_frame_display)

Output hidden; open in https://colab.research.google.com to view.

## CamShift 추적 메인 루프

### Back Projection (역투영) 개념
```
[전체 프레임 HSV] + [ROI 히스토그램]
        ↓  calcBackProject
[확률 맵(dst)] : 각 픽셀이 ROI 색상과 얼마나 유사한지 0~255로 표현
        ↓  × 마스크
[노이즈 제거된 확률 맵]
        ↓  CamShift
[회전된 추적 박스 반환]
```

### cv2.CamShift() 반환값
| 반환값 | 타입 | 내용 |
|--------|------|------|
| `ret` (rotated_rect) | RotatedRect | 중심점(cx,cy), 크기(w,h), 회전각(angle) |
| `track_window` | tuple | 다음 프레임 입력용 일반 bbox (x, y, w, h) |

In [28]:
# CamShift 추적 시작

# --- 유효성 검사 ---
if roi_hist is None:
    raise RuntimeError('roi_hist가 없습니다.')
if track_window is None:
    raise RuntimeError('track_window가 없습니다.')

print(f'CamShift 추적 시작 (최대 {MAX_FRAME_TO_PROCESS}프레임, {DISPLAY_EVERY_N_FRAMES}프레임마다 출력)...')

frame_count = 0

while cap.isOpened() and frame_count < MAX_FRAME_TO_PROCESS:
    ret_read, frame  = cap.read()
    if not ret_read:
        print(f'영상 끝(처리 완료된 프레임 : {frame_count})')
        break

    img_draw = frame.copy()

    # BGR -> HSV변환

    frame_hsv= cv2.cvtColor(frame, cv2.COLOR_BGR2HSV)

    # 마스크 생성

    target_mask = cv2.inRange(frame_hsv, np.array([0.,50.,50.]),np.array([180.,255.,255.]))

    #역투영
    # 전체 프레임에서 ROI 히스토그램 유사한 픽셀의 확률을 계산 -> 이게 역투영
    # dst : 단일 채널 (uint8) 확률 맵, 값이 클수록 ROI의 색상과 유사
    dst = cv2.calcBackProject([frame_hsv], [0], roi_hist, [0,180],1)

    # cv2.backProject([기준이미지],[채널],기준 히스토그램, Hue 값 범위, 스케일 )


    # 마스크 적용 : noise 영역의 확률을 0으로 만들기
    # 중요한건 dtype가 불일치 하면 오류나기 때문에 명시적으로 해줘야 함 uint8로 변환해서 곱셈

    dst = cv2.bitwise_and(dst,dst,mask=target_mask)
    # and를 하려면 형태가 같아야 하므로 얘로 명시한다?
    # 자기 자신과 and하니까 원본 그대로 인데. mask만 적용한다?

    # 자기자신 and + mask면
    # 마스크 값이 0이면 결과물은 모두 0
    # 마스크값이 255면 결과물은 원본 그대로

    # CamShift 추적 실행

    rotated_rect, track_window = cv2.CamShift(dst, track_window, termination)
    # rotated_rect (중심정(cx,ct),(w,h),angle) > 회전된 박스 정보
    # track_window (x,y,w,h) > 다음 프레임 기준 윈도우로 사용됨. 좌표값인데 그냥.

    # 회전된 추적 박스 시각화
    pts = cv2.boxPoints(rotated_rect) # 4개 꼭짓점 좌표 계산
    pts = np.int32(pts) # 정수로 변환 -> 그림그려야 하니까?
    cv2.polylines(img_draw,[pts],True, (0,255,0))
    # cv2.polylines(이미지, 점 좌표, 닫힌 도형인지?, 색깔)
    # 선 두께가 없네? 자동으로 채우나?


    # 중심점 표시
    cx,cy = int(rotated_rect[0][0]),int(rotated_rect[0][1])

    cv2.circle(img_draw, (cx,cy),4,(0,255,0),-1)


    # 프레임 번호 표시
    cv2.putText(img_draw, f'Frame : {frame_count}',(10,30),
                cv2.FONT_HERSHEY_SIMPLEX,0.8,(255,255,0), 2, cv2.LINE_AA)

    # 컬러 영상/ 역투영 영상 나란히 출력.
    dst_bgr = cv2.cvtColor(dst, cv2.COLOR_GRAY2BGR)
    result = np.hstack((img_draw,dst_bgr))

    # 정해진 간격으로 프레임마다 결과 표시
    if frame_count % DISPLAY_EVERY_N_FRAMES == 0:
        print(f'프레임 {frame_count} | 추적 중심: ({cx}, {cy})')
        cv2_imshow(result)
        time.sleep(1)


    frame_count += 1

cap.release()
# cv2.destroyAllWindows()

print(f'추적완료 | 처리한 프레임 : {frame_count}')


# 제가 뭔가를 잘못하기는 했는데요.// 강사님이 하신것도 잘 안되긴 했어요,
# 그래서 다른 알고리즘을 주로 쓴대요


Output hidden; open in https://colab.research.google.com to view.